# ETL Pipeline using AWS RDS## Extract, Transform, Load - Complete Guide### Resources- **Code**: https://colab.research.google.com/drive/1qBH_ZfTanr4N9QPQHfH0k9lbHpRa6EMx?usp=sharing- **Dataset**: https://www.kaggle.com/datasets/patrickb1912/ipl-complete-dataset-20082020- **Dream11 Points**: https://www.dream11.com/games/point-system---

## What is ETL?**ETL** stands for **Extract, Transform, Load** - a data integration process that combines data from multiple sources into a single, consistent data store.```┌─────────────────────────────────────────────────────────────┐│                      ETL PIPELINE                           │├─────────────────────────────────────────────────────────────┤│                                                             ││  ┌──────────┐    ┌──────────────┐    ┌──────────────┐      ││  │ EXTRACT  │ →  │  TRANSFORM   │ →  │     LOAD     │      ││  │          │    │              │    │              │      ││  │ CSV File │    │ Clean Data   │    │  AWS RDS     │      ││  │ API      │    │ Filter Rows  │    │  MySQL       │      ││  │ Database │    │ Add Columns  │    │  PostgreSQL  │      ││  │ Web      │    │ Aggregate    │    │  Data Warehouse│    ││  └──────────┘    └──────────────┘    └──────────────┘      ││                                                             │└─────────────────────────────────────────────────────────────┘```---

## The Three Steps### 1. Extract- Pull data from **source systems**- Sources: CSV, APIs, databases, web scraping- Keep original data intact### 2. Transform- **Clean** data (remove duplicates, handle nulls)- **Filter** relevant rows/columns- **Aggregate** data (groupby, sum, mean)- **Feature engineering** (create new columns)- **Validate** data quality### 3. Load- Write transformed data to **destination**- Destination: Data warehouse, database, data lake- Choose: Full load vs incremental load---

## Why ETL?| Benefit | Description ||---------|-------------|| **Data Integration** | Combine data from multiple sources || **Data Quality** | Clean and validate data || **Historical Analysis** | Store data for trend analysis || **Reporting** | Enable BI and analytics || **Machine Learning** | Prepare data for models || **Compliance** | Maintain data governance |---

---## What is AWS RDS?**Amazon RDS (Relational Database Service)** is a managed cloud database service that makes it easy to set up, operate, and scale a relational database in the cloud.### Supported Database Engines| Engine | Description ||--------|-------------|| **MySQL** | Most popular open-source DB || **PostgreSQL** | Advanced open-source DB || **MariaDB** | MySQL fork || **Oracle** | Enterprise DB || **SQL Server** | Microsoft DB || **Amazon Aurora** | AWS-optimized MySQL/PostgreSQL |### Why Use AWS RDS?- **Managed Service**: AWS handles backups, patching, monitoring- **Scalable**: Easy to resize storage and compute- **High Availability**: Multi-AZ deployments- **Security**: Encryption, VPC, IAM authentication- **Cost-Effective**: Pay only for what you use (Free Tier available)---

## Setting Up AWS RDS (Step-by-Step)### Prerequisites1. AWS Account (Free Tier eligible)2. MySQL Workbench or any SQL client### Steps1. **Login** to AWS Console2. **Search** for RDS3. **Click** "Create Database"4. **Choose** MySQL engine5. **Select** Free Tier template6. **Configure**:   - DB instance identifier: `ipl-database`   - Master username: `admin`   - Master password: `your_password`7. **Enable** Publicly Access (for testing)8. **Create** Security Group rule (port 3306)9. **Click** "Create Database"### Connection Details to Note```Endpoint: ipl-database.xxxxx.sa-east-1.rds.amazonaws.comPort: 3306Database: ipl_dbUsername: adminPassword: your_password```---

---## Installation```bash# Install required librariespip install pandas numpy mysql-connector-python sqlalchemy pymysql```---

In [ ]:
# Install required libraries# !pip install pandas numpy mysql-connector-python sqlalchemy pymysqlimport pandas as pdimport numpy as npimport mysql.connectorfrom sqlalchemy import create_engineimport warningswarnings.filterwarnings('ignore')print("Libraries imported successfully!")

---# Phase 1: EXTRACTThe Extract phase involves pulling raw data from the source system.### Data Sources for This Project- **matches.csv** - IPL match-level data (2008-2020)- **deliveries.csv** - Ball-by-ball delivery data---

## Download DatasetDownload the IPL Complete Dataset from Kaggle:https://www.kaggle.com/datasets/patrickb1912/ipl-complete-dataset-20082020---

In [ ]:
# Load the datasetsmatches = pd.read_csv('matches.csv')deliveries = pd.read_csv('deliveries.csv')print(f"Matches shape: {matches.shape}")print(f"Deliveries shape: {deliveries.shape}")

In [ ]:
# Inspect matches datamatches.head()

In [ ]:
# Inspect deliveries datadeliveries.head()

In [ ]:
# Check columnsprint("Matches columns:")print(matches.columns.tolist())print("\nDeliveries columns:")print(deliveries.columns.tolist())

---# Phase 2: TRANSFORMThe Transform phase involves cleaning, filtering, and enriching the data.### Transformations We'll Do1. **Clean** missing values2. **Filter** relevant columns3. **Feature Engineering** - Dream11 points4. **Aggregation** - Player stats5. **Data Type** conversions---

## Dream11 Point SystemDream11 awards points based on player performance:### Batting Points| Action | Points ||--------|--------|| Run | +1 || Boundary (4) | +1 bonus || Six (6) | +2 bonus || 30 runs | +4 bonus || 50 runs | +8 bonus || 100 runs | +16 bonus || Duck (0 runs, out) | -2 |### Bowling Points| Action | Points ||--------|--------|| Wicket | +25 || 3-wicket haul | +4 || 4-wicket haul | +8 || 5-wicket haul | +16 || Maiden over | +12 || Economy < 5 | +6 || Economy 5-6 | +4 || Economy 6-7 | +2 |### Other Points| Action | Points ||--------|--------|| Catch | +8 || Stumping | +12 || Run Out | +12 || Playing XI | +4 |---

## Step 1: Clean Matches Data

In [ ]:
# Check missing values in matchesprint("Missing values in matches:")print(matches.isnull().sum())

In [ ]:
# Drop columns with too many missing valuesmatches_clean = matches.drop(['umpire3'], axis=1, errors='ignore')# Drop rows with critical missing valuesmatches_clean = matches_clean.dropna(subset=['winner'])print(f"Cleaned matches shape: {matches_clean.shape}")

## Step 2: Clean Deliveries Data

In [ ]:
# Check missing values in deliveriesprint("Missing values in deliveries:")print(deliveries.isnull().sum())

In [ ]:
# Fill missing values in player_dismisseddeliveries_clean = deliveries.copy()deliveries_clean['player_dismissed'] = deliveries_clean['player_dismissed'].fillna('None')deliveries_clean['dismissal_kind'] = deliveries_clean['dismissal_kind'].fillna('None')deliveries_clean['fielder'] = deliveries_clean['fielder'].fillna('None')print(f"Cleaned deliveries shape: {deliveries_clean.shape}")

## Step 3: Feature Engineering - Dream11 Points### Batting Points Calculation

In [ ]:
# Calculate batting points for each balldef calculate_batting_points(row):    points = 0    # Run points    points += row['batsman_runs']    # Boundary bonus    if row['batsman_runs'] == 4:        points += 1    elif row['batsman_runs'] == 6:        points += 2    return points# Apply functiondeliveries_clean['batting_points'] = deliveries_clean.apply(calculate_batting_points, axis=1)print("Batting points calculated!")deliveries_clean[['batsman', 'batsman_runs', 'batting_points']].head(10)

### Bowling Points Calculation

In [ ]:
# Calculate bowling points for each balldef calculate_bowling_points(row):    points = 0    # Wicket points    if row['player_dismissed'] != 'None':        points += 25    # Extra runs (wides and no-balls count against bowler)    if row['wide_runs'] > 0:        points -= 1    if row['noball_runs'] > 0:        points -= 1    return points# Apply functiondeliveries_clean['bowling_points'] = deliveries_clean.apply(calculate_bowling_points, axis=1)print("Bowling points calculated!")deliveries_clean[['bowler', 'player_dismissed', 'bowling_points']].head(10)

### Fielding Points Calculation

In [ ]:
# Calculate fielding pointsdef calculate_fielding_points(row):    points = 0    # Catch    if row['dismissal_kind'] == 'caught':        points += 8    # Stumping    elif row['dismissal_kind'] == 'stumped':        points += 12    # Run out    elif row['dismissal_kind'] == 'run out':        points += 12    return points# Apply functiondeliveries_clean['fielding_points'] = deliveries_clean.apply(calculate_fielding_points, axis=1)print("Fielding points calculated!")

### Total Dream11 Points

In [ ]:
# Calculate total points per balldeliveries_clean['total_points'] = (    deliveries_clean['batting_points'] +    deliveries_clean['bowling_points'] +    deliveries_clean['fielding_points'])print("Total points calculated!")deliveries_clean[['batsman', 'bowler', 'batting_points', 'bowling_points', 'fielding_points', 'total_points']].head(10)

## Step 4: Player Match SummaryAggregate points per player per match.---

In [ ]:
# Create player match summaryplayer_match = deliveries_clean.groupby(['match_id', 'batsman']).agg(    runs=('batsman_runs', 'sum'),    balls_faced=('ball', 'count'),    fours=('batsman_runs', lambda x: (x == 4).sum()),    sixes=('batsman_runs', lambda x: (x == 6).sum()),    batting_pts=('batting_points', 'sum')).reset_index()# Calculate strike rateplayer_match['strike_rate'] = (player_match['runs'] / player_match['balls_faced'] * 100).round(2)# Add milestone bonusesplayer_match['milestone_bonus'] = 0player_match.loc[player_match['runs'] >= 30, 'milestone_bonus'] += 4player_match.loc[player_match['runs'] >= 50, 'milestone_bonus'] += 8player_match.loc[player_match['runs'] >= 100, 'milestone_bonus'] += 16# Duck penalty (0 runs and out)ducks = deliveries_clean[    (deliveries_clean['batsman_runs'] == 0) &    (deliveries_clean['player_dismissed'] != 'None')].groupby(['match_id', 'batsman']).size().reset_index(name='duck_count')player_match = player_match.merge(ducks, on=['match_id', 'batsman'], how='left')player_match['duck_count'] = player_match['duck_count'].fillna(0)player_match['duck_penalty'] = player_match['duck_count'] * -2# Final batting pointsplayer_match['final_batting_points'] = (    player_match['batting_pts'] +    player_match['milestone_bonus'] +    player_match['duck_penalty'])print(f"Player match summary shape: {player_match.shape}")player_match.head()

## Step 5: Bowling Summary

In [ ]:
# Create bowling summarybowling_match = deliveries_clean.groupby(['match_id', 'bowler']).agg(    balls_bowled=('ball', 'count'),    runs_conceded=('total_runs', 'sum'),    wickets=('player_dismissed', lambda x: (x != 'None').sum()),    wides=('wide_runs', 'sum'),    noballs=('noball_runs', 'sum'),    bowling_pts=('bowling_points', 'sum')).reset_index()# Calculate oversbowling_match['overs'] = (bowling_match['balls_bowled'] / 6).round(2)# Calculate economy ratebowling_match['economy'] = (bowling_match['runs_conceded'] / bowling_match['overs']).round(2)# Economy bonusbowling_match['economy_bonus'] = 0bowling_match.loc[bowling_match['economy'] < 5, 'economy_bonus'] = 6bowling_match.loc[(bowling_match['economy'] >= 5) & (bowling_match['economy'] < 6), 'economy_bonus'] = 4bowling_match.loc[(bowling_match['economy'] >= 6) & (bowling_match['economy'] < 7), 'economy_bonus'] = 2# Wicket haul bonusesbowling_match['haul_bonus'] = 0bowling_match.loc[bowling_match['wickets'] >= 3, 'haul_bonus'] += 4bowling_match.loc[bowling_match['wickets'] >= 4, 'haul_bonus'] += 8bowling_match.loc[bowling_match['wickets'] >= 5, 'haul_bonus'] += 16# Final bowling pointsbowling_match['final_bowling_points'] = (    bowling_match['bowling_pts'] +    bowling_match['economy_bonus'] +    bowling_match['haul_bonus'])print(f"Bowling match summary shape: {bowling_match.shape}")bowling_match.head()

## Step 6: Merge with Match Data

In [ ]:
# Add match details to player summaryplayer_summary = player_match.merge(    matches_clean[['id', 'season', 'city', 'date', 'team1', 'team2', 'winner']],    left_on='match_id',    right_on='id',    how='left')# Add playing XI bonus (4 points for each player in the match)player_summary['playing_xi_bonus'] = 4# Final Dream11 pointsplayer_summary['dream11_points'] = (    player_summary['final_batting_points'] +    player_summary['playing_xi_bonus'])print(f"Final player summary shape: {player_summary.shape}")player_summary.head()

## Step 7: Overall Player Stats

In [ ]:
# Calculate overall player statisticsoverall_stats = player_summary.groupby('batsman').agg(    total_matches=('match_id', 'nunique'),    total_runs=('runs', 'sum'),    total_balls=('balls_faced', 'sum'),    total_fours=('fours', 'sum'),    total_sixes=('sixes', 'sum'),    total_dream11_points=('dream11_points', 'sum'),    avg_dream11_points=('dream11_points', 'mean')).reset_index()# Calculate overall strike rateoverall_stats['overall_strike_rate'] = (    overall_stats['total_runs'] / overall_stats['total_balls'] * 100).round(2)# Sort by total runsoverall_stats = overall_stats.sort_values('total_runs', ascending=False)print(f"Overall stats shape: {overall_stats.shape}")overall_stats.head(10)

---# Phase 3: LOADThe Load phase involves writing the transformed data to the destination database.### Destination: AWS RDS MySQL---

## Database Connection Setup### Option 1: Using mysql-connector

In [ ]:
# Database connection details# Replace with your actual RDS credentialsDB_CONFIG = {    'host': 'your-rds-endpoint.rds.amazonaws.com',  # Your RDS endpoint    'user': 'admin',                                  # Master username    'password': 'your_password',                      # Master password    'database': 'ipl_db',                             # Database name    'port': 3306                                      # Default MySQL port}# Test connectiontry:    conn = mysql.connector.connect(**DB_CONFIG)    print("Connection successful!")    conn.close()except Exception as e:    print(f"Connection failed: {e}")

### Option 2: Using SQLAlchemy (Recommended)

In [ ]:
# SQLAlchemy connection (recommended for pandas)# Replace with your actual RDS credentialsRDS_ENDPOINT = 'your-rds-endpoint.rds.amazonaws.com'RDS_USER = 'admin'RDS_PASSWORD = 'your_password'RDS_DB = 'ipl_db'RDS_PORT = 3306# Create engineengine = create_engine(    f'mysql+pymysql://{RDS_USER}:{RDS_PASSWORD}@{RDS_ENDPOINT}:{RDS_PORT}/{RDS_DB}')print("SQLAlchemy engine created!")

## Create Tables in RDS

In [ ]:
# Create tables using SQLcreate_tables_sql = (    "-- Matches table\n"    "CREATE TABLE IF NOT EXISTS matches (\n"    "    id INT PRIMARY KEY,\n"    "    season INT,\n"    "    city VARCHAR(100),\n"    "    date DATE,\n"    "    team1 VARCHAR(100),\n"    "    team2 VARCHAR(100),\n"    "    toss_winner VARCHAR(100),\n"    "    toss_decision VARCHAR(20),\n"    "    result VARCHAR(20),\n"    "    dl_applied INT,\n"    "    winner VARCHAR(100),\n"    "    win_by_runs INT,\n"    "    win_by_wickets INT,\n"    "    player_of_match VARCHAR(100),\n"    "    venue VARCHAR(200),\n"    "    umpire1 VARCHAR(100),\n"    "    umpire2 VARCHAR(100)\n"    ");\n\n"    "-- Deliveries table\n"    "CREATE TABLE IF NOT EXISTS deliveries (\n"    "    match_id INT,\n"    "    inning INT,\n"    "    batting_team VARCHAR(100),\n"    "    bowling_team VARCHAR(100),\n"    "    `over` INT,\n"    "    ball INT,\n"    "    batsman VARCHAR(100),\n"    "    non_striker VARCHAR(100),\n"    "    bowler VARCHAR(100),\n"    "    is_super_over INT,\n"    "    wide_runs INT,\n"    "    bye_runs INT,\n"    "    legbye_runs INT,\n"    "    noball_runs INT,\n"    "    penalty_runs INT,\n"    "    batsman_runs INT,\n"    "    extra_runs INT,\n"    "    total_runs INT,\n"    "    player_dismissed VARCHAR(100),\n"    "    dismissal_kind VARCHAR(50),\n"    "    fielder VARCHAR(100)\n"    ");\n\n"    "-- Dream11 points table\n"    "CREATE TABLE IF NOT EXISTS dream11_points (\n"    "    match_id INT,\n"    "    batsman VARCHAR(100),\n"    "    runs INT,\n"    "    balls_faced INT,\n"    "    fours INT,\n"    "    sixes INT,\n"    "    strike_rate FLOAT,\n"    "    dream11_points INT,\n"    "    season INT,\n"    "    team VARCHAR(100)\n"    ");\n\n"    "-- Player overall stats table\n"    "CREATE TABLE IF NOT EXISTS player_stats (\n"    "    batsman VARCHAR(100) PRIMARY KEY,\n"    "    total_matches INT,\n"    "    total_runs INT,\n"    "    total_balls INT,\n"    "    total_fours INT,\n"    "    total_sixes INT,\n"    "    overall_strike_rate FLOAT,\n"    "    total_dream11_points INT,\n"    "    avg_dream11_points FLOAT\n"    ");")print("SQL tables defined!")

## Load Data to RDS

In [ ]:
# Load matches dataprint("Loading matches data...")matches_clean.to_sql(    'matches',    con=engine,    if_exists='replace',    index=False)print(f"Loaded {len(matches_clean)} matches")

In [ ]:
# Load deliveries data (in chunks for large datasets)print("Loading deliveries data...")chunk_size = 10000for i in range(0, len(deliveries_clean), chunk_size):    chunk = deliveries_clean.iloc[i:i+chunk_size]    chunk.to_sql(        'deliveries',        con=engine,        if_exists='append' if i > 0 else 'replace',        index=False    )    print(f"  Loaded chunk {i//chunk_size + 1}")print(f"Loaded {len(deliveries_clean)} deliveries")

In [ ]:
# Load dream11 pointsprint("Loading Dream11 points...")player_summary[['match_id', 'batsman', 'runs', 'balls_faced', 'fours', 'sixes',                'strike_rate', 'dream11_points', 'season', 'team1']].to_sql(    'dream11_points',    con=engine,    if_exists='replace',    index=False)print(f"Loaded {len(player_summary)} Dream11 records")

In [ ]:
# Load overall player statsprint("Loading player stats...")overall_stats.to_sql(    'player_stats',    con=engine,    if_exists='replace',    index=False)print(f"Loaded {len(overall_stats)} player stats")

---# Querying Data from AWS RDSNow let's verify the data and run analytical queries.---

In [ ]:
# Read data back from RDSquery = "SELECT * FROM player_stats ORDER BY total_runs DESC LIMIT 10"top_batsmen = pd.read_sql(query, con=engine)top_batsmen

In [ ]:
# Top Dream11 point scorersquery = (    "SELECT batsman, total_matches, total_runs, total_dream11_points, "    "ROUND(avg_dream11_points, 2) as avg_points_per_match "    "FROM player_stats "    "ORDER BY total_dream11_points DESC "    "LIMIT 10")top_dream11 = pd.read_sql(query, con=engine)top_dream11

In [ ]:
# Best economy bowlers (minimum 50 overs)query = (    "SELECT bowler, COUNT(DISTINCT match_id) as matches, "    "SUM(wickets) as total_wickets, "    "ROUND(AVG(economy), 2) as avg_economy "    "FROM ("    "SELECT match_id, bowler, "    "SUM(total_runs) as runs_conceded, "    "COUNT(*)/6 as overs, "    "SUM(total_runs)/(COUNT(*)/6) as economy, "    "SUM(CASE WHEN player_dismissed != 'None' THEN 1 ELSE 0 END) as wickets "    "FROM deliveries "    "GROUP BY match_id, bowler "    "HAVING COUNT(*) >= 12"    ") subq "    "GROUP BY bowler "    "HAVING SUM(overs) >= 50 "    "ORDER BY avg_economy ASC "    "LIMIT 10")best_economy = pd.read_sql(query, con=engine)best_economy

In [ ]:
# Matches per seasonquery = (    "SELECT season, COUNT(*) as total_matches, "    "COUNT(DISTINCT winner) as unique_winners "    "FROM matches "    "GROUP BY season "    "ORDER BY season")season_stats = pd.read_sql(query, con=engine)season_stats

---# ETL Pipeline Summary## What We Built```┌─────────────────────────────────────────────────────────────┐│                    ETL PIPELINE FLOW                        │├─────────────────────────────────────────────────────────────┤│                                                             ││  ┌─────────────┐                                           ││  │  CSV Files  │  ← IPL Dataset (matches.csv, deliveries.csv)││  └──────┬──────┘                                           ││         │                                                   ││         ▼                                                   ││  ┌─────────────┐                                           ││  │   EXTRACT   │  ← pd.read_csv()                          ││  └──────┬──────┘                                           ││         │                                                   ││         ▼                                                   ││  ┌─────────────┐                                           ││  │  TRANSFORM  │  ← Clean, Filter, Feature Engineering     ││  │             │  ← Dream11 Points Calculation             ││  │             │  ← Aggregation & Summary                  ││  └──────┬──────┘                                           ││         │                                                   ││         ▼                                                   ││  ┌─────────────┐                                           ││  │    LOAD     │  ← AWS RDS MySQL                          ││  │             │  ← 4 Tables: matches, deliveries,         ││  │             │    dream11_points, player_stats            ││  └──────┬──────┘                                           ││         │                                                   ││         ▼                                                   ││  ┌─────────────┐                                           ││  │   QUERY     │  ← Analytics & Reporting                  ││  └─────────────┘                                           ││                                                             │└─────────────────────────────────────────────────────────────┘```## Tables Created| Table | Description | Rows ||-------|-------------|------|| `matches` | Match-level data | ~750 || `deliveries` | Ball-by-ball data | ~190K || `dream11_points` | Player Dream11 points | ~25K || `player_stats` | Overall player stats | ~500 |## Key Transformations1. **Dream11 Points**: Calculated batting, bowling, fielding points2. **Milestone Bonuses**: 30+, 50+, 100+ runs3. **Economy Bonus**: Based on bowling economy rate4. **Duck Penalty**: -2 points for ducks5. **Playing XI Bonus**: +4 points for each player## Technologies Used| Technology | Purpose ||------------|---------|| **Python** | Core language || **Pandas** | Data transformation || **MySQL** | Database engine || **AWS RDS** | Cloud database || **SQLAlchemy** | Database connectivity |---## Next Steps1. **Schedule** ETL pipeline with AWS Lambda2. **Add** real-time data sources3. **Build** dashboards with the data4. **Implement** incremental loads5. **Add** data quality checks---